# Build the network of a region

scgraph publishes networks of the whole world, ready to load (see the notebook *Routes on a multimodal network*). Their data is that of the day of their release.

To get the data of today for one region, build its network on your machine. One call does it: the pipeline `geofabrik_multimodal` reads the newest OpenStreetMap extract of the region. The build is also the place to start from when you want rules of your own, for example other kinds of roads.

This notebook builds the network of Denmark:

1. Build the network with one call.
2. Read what the network file says about itself.
3. Compare it with the released network of Europe.
4. Compute routes on some modes.
5. Check the network against known routes.

The build downloads about 520 MB, mostly the OpenStreetMap extract of Denmark. It takes about one minute after the downloads, and about 2 GB of memory.

GitHub does not show the maps of a notebook. To see them, run the notebook, or open it in [nbviewer](https://nbviewer.org/github/connor-makowski/scgraph/blob/main/examples/build_a_region.ipynb).

## Setup

The build reads OpenStreetMap files with `osmium`, which the extra `osm` of scgraph installs. The maps need [folium](https://python-visualization.github.io/folium/).

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph[osm] @ git+https://github.com/connor-makowski/scgraph@dev" folium
import collections
import logging
import os

import folium

from scgraph import GeoGraph
from scgraph.pipelines import geofabrik_multimodal
from scgraph.pipelines.build import build_file
from scgraph.pipelines.steps.stage4_validation import load_cases, validate_network

## 1. Build the network

`build_geograph` takes the name of a pipeline and a region. The region is the path of an extract on [Geofabrik](https://download.geofabrik.de), a server of OpenStreetMap extracts: `europe/denmark`, `europe/france/bretagne`, `north-america/us/texas`.

The first call downloads the sources and builds the network. It keeps the network in the cache of scgraph. scgraph reports its work through `logging`: the next cell shows these messages.

In [2]:
logging.basicConfig(format="%(message)s")
logging.getLogger("scgraph.pipelines").setLevel(logging.INFO)

denmark = GeoGraph.build_geograph("geofabrik_multimodal", region="europe/denmark")

logging.getLogger("scgraph.pipelines").setLevel(logging.WARNING)  # the cells below stay quiet

Building geofabrik_multimodal


Built geofabrik_multimodal. Credit: Eurostat, SeaRoute maritime network (from the Oak Ridge National Laboratory Global Shipping Lane Network, 2000, enriched with AIS-based lines around European coasts); Natural Earth (1:10m); National Geospatial-Intelligence Agency, World Port Index (Pub. 150); OurAirports; The OpenSky Network, flight list (Strohmeier et al., Earth Syst. Sci. Data 13, 357-366, 2021); US Bureau of Transportation Statistics, T-100 Segment; Eurostat, air transport routes (avia_par, avia_gor); Wikidata; © OpenStreetMap contributors (EUPL-1.2; LicenseRef-public-domain; LicenseRef-US-Government-Work; CC-BY-4.0; CC0-1.0; ODbL-1.0)


A later call loads the network from the cache. It makes no download and builds nothing:

In [3]:
denmark = GeoGraph.build_geograph("geofabrik_multimodal", region="europe/denmark")
print(f"{len(denmark.nodes):,} nodes")

50,321 nodes


## 2. What the network file says about itself

The build wrote one network file. Its header records the pipeline, the sources with the date of their data, and the credit to cite. `build_file` gives the path of the file, and `get_scgraph_info` reads its header.

In [4]:
file = build_file("geofabrik_multimodal", region="europe/denmark")
info = GeoGraph.get_scgraph_info(file)

print(f"File: {os.path.basename(file)[:16]}....scgraph, {os.path.getsize(file) / 1e6:.1f} MB")
print(f"{info['nodes']:,} nodes and {info['arcs']:,} arcs. The lengths are in {info['units']}.")
print("Pipeline:", info["pipeline"])
print("Newest source data:", info["data_up_to"])
print()
for source in info["sources"]:
    print(f"    {source['name']:<15}{source.get('version') or '':<45.44}{source['credit'][:40]}")
print()
print("Credit to cite with your results:", info["attribution"])

File: 4537b7eff4e69229....scgraph, 4.2 MB
50,321 nodes and 172,000 arcs. The lengths are in km.
Pipeline: {'name': 'geofabrik_multimodal', 'region': 'europe/denmark', 'version': 2}
Newest source data: 2026-11-19T21:22:20Z

    sea                                                         Eurostat, SeaRoute maritime network (fro
    land           5.1.2                                        Natural Earth (1:10m)
    lakes          5.1.2                                        Natural Earth (1:10m)
    rivers         5.1.2                                        Natural Earth (1:10m)
    ports          UpdatedPub150.csv, SHA-256 315644f1e7796629  National Geospatial-Intelligence Agency,
    airports       airports.csv at commit 1cdbf9cbed1e47bed8f11 OurAirports
    flights        flights of 2021: OpenSky v21.12; T-100 and E The OpenSky Network, flight list (Strohm
    rail_terminals query of 2026-11-16                          Wikidata
    osm            2026-11-19T21:22:20Z                

The name of the file is the SHA-256 digest of its content. The Geofabrik extract changes every day, so a build on another day gives another network. Record this digest, or keep the file, when a result must be found again later.

Each node has a mode: the network that it belongs to, or the kind of hub that it is.

In [5]:
mode_names = [mode["name"] for mode in denmark.modes]
nodes_per_mode = collections.Counter(mode_names[row] for row in denmark.node_modes)
for name in mode_names:
    print(f"{name:<14}{nodes_per_mode[name]:>7,} nodes")

road            6,163 nodes
rail            1,927 nodes
sea            42,157 nodes
port               63 nodes
airport            10 nodes
rail_terminal       1 nodes


The roads and the railways are those of Denmark. The sea lanes are those of the whole world, so a route can leave Denmark by sea. A port, an airport or a rail terminal is a hub: a place where a route changes network.

## 3. Compare with the released network

The released network `europe` covers Denmark too. The two networks hold the same kinds of roads. They differ in two ways:

- **The date of the data.** A released network has the data of its release. The network built here has the data of the newest extract.
- **The nodes along a road.** Both networks have a node at each junction. Along a road without a junction, the released network has a node at most about every 20 km. The network built here has one at most about every 4 km.

The next cell reads the date of the data of the two networks, and counts their road nodes in the same rectangle around Denmark.

In [6]:
europe = GeoGraph.load_geograph("europe", modes=["road"], shapes=False)
denmark_roads = GeoGraph.load_from_scgraph(file, modes=["road"])

SOUTH, WEST, NORTH, EAST = 54.5, 8.0, 57.8, 12.8  # Denmark without the island of Bornholm


def get_nodes_inside(geograph, south, west, north, east):
    """The nodes of a geograph between two latitudes and two longitudes."""
    return [
        point
        for point in geograph.nodes
        if south <= point[0] <= north and west <= point[1] <= east
    ]


released_nodes = get_nodes_inside(europe, SOUTH, WEST, NORTH, EAST)
built_nodes = get_nodes_inside(denmark_roads, SOUTH, WEST, NORTH, EAST)
released_info = GeoGraph.get_scgraph_info(europe.scgraph_file)
print(f"Released network `europe`: data up to {released_info['data_up_to']}, {len(released_nodes):,} road nodes in the rectangle")
print(f"Network built here:        data up to {info['data_up_to']}, {len(built_nodes):,} road nodes in the rectangle")

Released network `europe`: data up to 2026-11-02T00:59:53Z, 5,681 road nodes in the rectangle
Network built here:        data up to 2026-11-19T21:22:20Z, 6,127 road nodes in the rectangle


The map shows the road nodes of the two networks around Aarhus. A large black circle is a node of the released network. A small orange dot is a node of the network built here.

In [7]:
def make_map(south, west, north, east, height=440):
    """A new map that shows the area between two latitudes and two longitudes."""
    new_map = folium.Map(tiles=None)
    # `control=False` keeps the background out of the control that shows or hides layers
    folium.TileLayer("OpenStreetMap", control=False).add_to(new_map)
    new_map.fit_bounds([[south, west], [north, east]])
    # The figure gives the map its height in the notebook
    folium.Figure(height=height).add_child(new_map)
    return new_map


AARHUS_AREA = (56.0, 9.9, 56.3, 10.5)

node_map = make_map(*AARHUS_AREA)
for point in get_nodes_inside(denmark_roads, *AARHUS_AREA):
    folium.CircleMarker(point, radius=2, color="#eb6834", fill_opacity=1).add_to(node_map)
for point in get_nodes_inside(europe, *AARHUS_AREA):
    folium.CircleMarker(point, radius=8, color="#0b0b0b", weight=2, fill=False).add_to(node_map)
node_map

The two networks have the same junctions. The network built here has more nodes along the roads between two junctions. In a country with many junctions, such as Denmark, the two counts are close.

## 4. Routes on some modes

A route is the shortest in kilometres. With all the modes, it can take a railway, a sea lane or a flight. `modes` loads some modes of the file only: here the roads, which include the vehicle ferries.

The example goes from Aalborg, in the north of Denmark, to Copenhagen.

In [8]:
AALBORG = {"latitude": 57.0488, "longitude": 9.9217}
COPENHAGEN = {"latitude": 55.6761, "longitude": 12.5683}


def get_modes_of(geograph, route):
    """The names of the modes of the nodes of a route, in the order of the modes table."""
    names = [mode["name"] for mode in geograph.modes]
    rows = sorted({geograph.node_modes[node] for node in route["path"]})
    return [names[row] for row in rows]


routes = {}
for title, geograph in [("All modes", denmark), ('modes=["road"]', denmark_roads)]:
    route = geograph.get_shortest_path(
        origin_node=AALBORG, destination_node=COPENHAGEN, output_path=True
    )
    routes[title] = route
    print(f"{title}: {route['length']:,.1f} km, through nodes of the modes {get_modes_of(geograph, route)}")

All modes: 261.1 km, through nodes of the modes ['rail', 'sea', 'port']
modes=["road"]: 321.6 km, through nodes of the modes ['road']


In [9]:
COLORS = {"All modes": "#0b0b0b", 'modes=["road"]': "#eb6834"}

route_map = make_map(55.3, 9.0, 57.3, 13.0, height=520)
for title, route in routes.items():
    layer = folium.FeatureGroup(name=f"{title}: {route['length']:,.1f} km")
    folium.PolyLine(
        route["coordinate_path"],
        color=COLORS[title],
        weight=4,
        tooltip=f"{title}: {route['length']:,.1f} km",
    ).add_to(layer)
    layer.add_to(route_map)
for name, place in [("Aalborg", AALBORG), ("Copenhagen", COPENHAGEN)]:
    folium.CircleMarker(
        [place["latitude"], place["longitude"]],
        radius=4,
        color="#0b0b0b",
        fill_opacity=1,
        tooltip=folium.Tooltip(name, permanent=True, direction="top"),
    ).add_to(route_map)
folium.LayerControl(collapsed=False).add_to(route_map)
route_map

With all the modes, the shortest route goes by rail to a port of Aalborg, then by sea to Copenhagen. No traveller goes this way, but it is the shortest line in kilometres. By road, the route crosses the sea from Aarhus on a vehicle ferry, which is part of the mode `road`.

Choose the modes that fit your question. To compare modes by time or by cost, compute a cost for each mode from the kilometres. The notebook *Routes on a multimodal network* shows how to read the class of each arc of a route.

## 5. Check the network against known routes

A pipeline comes with **validation cases**. A case describes one fact of the real world that the network must reproduce: the length of a known route, or a hub that must join its networks. `validate_network` checks a network file against a list of cases.

The pipeline lists three cases for Denmark:

In [10]:
cases = load_cases(geofabrik_multimodal.VALIDATION, region="europe/denmark")
for case in cases:
    print(f"{case['id']:<34}{case['kind']:<15}{case['source'][:60]}")

result = validate_network(file, cases)
print()
print(result.summary())

aarhus_to_copenhagen_intermodal   route_lengths  Molslinjen / Geofabrik
arhus_port                        hubs           port: NGA World Port Index (Pub. 150, UpdatedPub150.csv SHA-
aalborg_airport                   hubs           airport: OurAirports (commit 1cdbf9cb), table airports.csv o

PASSED: 3/3 cases OK (failed: 0)


The same check runs at the end of a build with `build_file(..., validate=True)`, or with `--validate` on the command line. The network then enters the cache only if it passes its cases.

A region of your own has no case in scgraph. Write a few cases for it from routes that you know: the docstring of `scgraph.pipelines.steps.stage4_validation` gives the kinds of cases and their rules.

## What to remember

- `GeoGraph.build_geograph("geofabrik_multimodal", region=...)` builds the multimodal network of a Geofabrik extract, with the data of the day. It needs `pip install scgraph[osm]`.
- The build keeps the network in the cache. `build_file` gives the path of its file, and `get_scgraph_info` reads its sources and the credit to cite.
- The extract changes every day. Record the name of the file, which is the digest of its content, when a result must be found again.
- `load_from_scgraph(file, modes=[...])` loads some modes of the file.
- `validate_network` checks the network against known routes.

The guide [From open data to a route](https://github.com/connor-makowski/scgraph/blob/main/scgraph/pipelines/README.md), Part 1, follows this build step by step, with its numbers.